In [1]:
!pip install faker pandas numpy scikit-learn plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import random
import os

from faker import Faker
from datetime import datetime, timedelta

print("LedgerLens environment ready!")

LedgerLens environment ready!


In [3]:
os.makedirs("data", exist_ok=True)
os.makedirs("reports", exist_ok=True)
os.makedirs("knowledge", exist_ok=True)

print("Project folders created!")

Project folders created!


In [4]:
from faker import Faker
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

fake = Faker()

random.seed(42)
np.random.seed(42)

NUM_TRANSACTIONS = 2000

start_date = datetime(2026, 8, 1)

transactions = []

for i in range(NUM_TRANSACTIONS):

    transaction_id = f"TXN{i+1:06d}"
    order_id = f"ORD{i+1:06d}"
    customer_id = f"CUST{random.randint(1, 700):05d}"

    amount = round(
        random.uniform(100, 25000),
        2
    )

    payment_date = (
        start_date +
        timedelta(days=random.randint(0, 25))
    )

    transactions.append({
        "transaction_id": transaction_id,
        "order_id": order_id,
        "customer_id": customer_id,
        "amount": amount,
        "payment_date": payment_date.date(),
        "payment_status": "SUCCESS"
    })

transactions_df = pd.DataFrame(transactions)

transactions_df.head()

,transaction_id,order_id,customer_id,amount,payment_date,payment_status
0,TXN000001,ORD000001,CUST00655,2872.14,2026-08-24,SUCCESS
1,TXN000002,ORD000002,CUST00282,6197.81,2026-08-05,SUCCESS
2,TXN000003,ORD000003,CUST00105,16949.82,2026-08-18,SUCCESS
3,TXN000004,ORD000004,CUST00090,14803.26,2026-08-02,SUCCESS
4,TXN000005,ORD000005,CUST00031,2433.01,2026-08-08,SUCCESS


In [5]:
fees = []

for _, row in transactions_df.iterrows():

    fee_rate = random.uniform(0.01, 0.025)

    gateway_fee = round(
        row["amount"] * fee_rate,
        2
    )

    tax_on_fee = round(
        gateway_fee * 0.18,
        2
    )

    fees.append({
        "transaction_id": row["transaction_id"],
        "gateway_fee": gateway_fee,
        "tax_on_fee": tax_on_fee
    })

fees_df = pd.DataFrame(fees)

fees_df.head()

,transaction_id,gateway_fee,tax_on_fee
0,TXN000001,32.58,5.86
1,TXN000002,124.43,22.40
2,TXN000003,277.18,49.89
3,TXN000004,225.37,40.57
4,TXN000005,36.04,6.49


In [6]:
refunds = []

refund_counter = 1

for _, row in transactions_df.iterrows():

    # Approximately 8% of transactions receive refunds
    if random.random() < 0.08:

        refund_amount = round(
            row["amount"] *
            random.uniform(0.10, 1.00),
            2
        )

        refunds.append({
            "refund_id": f"REF{refund_counter:06d}",
            "transaction_id": row["transaction_id"],
            "refund_amount": refund_amount,
            "refund_status": "SUCCESS"
        })

        refund_counter += 1

refunds_df = pd.DataFrame(refunds)

print("Refund records:", len(refunds_df))

refunds_df.head()

Refund records: 152


,refund_id,transaction_id,refund_amount,refund_status
0,REF000001,TXN000002,2526.04,SUCCESS
1,REF000002,TXN000020,4159.84,SUCCESS
2,REF000003,TXN000021,1883.08,SUCCESS
3,REF000004,TXN000052,1543.99,SUCCESS
4,REF000005,TXN000068,340.07,SUCCESS


In [7]:
settlements = []

settlement_counter = 1

for _, transaction in transactions_df.iterrows():

    transaction_id = transaction["transaction_id"]

    fee_row = fees_df[
        fees_df["transaction_id"] == transaction_id
    ]

    gateway_fee = fee_row["gateway_fee"].iloc[0]
    tax_on_fee = fee_row["tax_on_fee"].iloc[0]

    refund_rows = refunds_df[
        refunds_df["transaction_id"] == transaction_id
    ]

    if len(refund_rows) > 0:
        refund_amount = refund_rows["refund_amount"].sum()
    else:
        refund_amount = 0

    expected_settlement = (
        transaction["amount"]
        - gateway_fee
        - tax_on_fee
        - refund_amount
    )

    expected_settlement = round(
        max(expected_settlement, 0),
        2
    )

    # Introduce realistic settlement problems

    random_value = random.random()

    # Missing settlement
    if random_value < 0.03:
        continue

    # Under-settlement
    elif random_value < 0.06:

        deduction = round(
            random.uniform(100, 1500),
            2
        )

        settled_amount = max(
            expected_settlement - deduction,
            0
        )

    # Normal settlement
    else:

        settled_amount = expected_settlement

    settlement_date = (
        pd.to_datetime(transaction["payment_date"])
        + pd.Timedelta(days=random.randint(1, 3))
    )

    settlements.append({
        "settlement_id":
            f"SET{settlement_counter:06d}",

        "transaction_id":
            transaction_id,

        "settled_amount":
            round(settled_amount, 2),

        "settlement_date":
            settlement_date.date()
    })

    settlement_counter += 1

settlements_df = pd.DataFrame(settlements)

print("Settlement records:", len(settlements_df))

settlements_df.head()

Settlement records: 1947


,settlement_id,transaction_id,settled_amount,settlement_date
0,SET000001,TXN000001,2833.70,2026-08-26
1,SET000002,TXN000002,3524.94,2026-08-08
2,SET000003,TXN000003,16622.75,2026-08-19
3,SET000004,TXN000004,14537.32,2026-08-03
4,SET000005,TXN000005,2390.48,2026-08-09


In [8]:
transactions_df.to_csv(
    "data/transactions.csv",
    index=False
)

fees_df.to_csv(
    "data/fees.csv",
    index=False
)

refunds_df.to_csv(
    "data/refunds.csv",
    index=False
)

settlements_df.to_csv(
    "data/settlements.csv",
    index=False
)

print("All datasets saved successfully!")

All datasets saved successfully!


In [9]:
reconciliation = transactions_df.merge(
    fees_df,
    on="transaction_id",
    how="left"
)

refund_summary = (
    refunds_df
    .groupby("transaction_id")["refund_amount"]
    .sum()
    .reset_index()
)

reconciliation = reconciliation.merge(
    refund_summary,
    on="transaction_id",
    how="left"
)

reconciliation["refund_amount"] = (
    reconciliation["refund_amount"]
    .fillna(0)
)

reconciliation = reconciliation.merge(
    settlements_df[
        [
            "transaction_id",
            "settled_amount",
            "settlement_id"
        ]
    ],
    on="transaction_id",
    how="left"
)

reconciliation["expected_settlement"] = (
    reconciliation["amount"]
    - reconciliation["gateway_fee"]
    - reconciliation["tax_on_fee"]
    - reconciliation["refund_amount"]
)

reconciliation["expected_settlement"] = (
    reconciliation["expected_settlement"]
    .clip(lower=0)
    .round(2)
)

reconciliation["difference"] = (
    reconciliation["expected_settlement"]
    - reconciliation["settled_amount"]
)

reconciliation["difference"] = (
    reconciliation["difference"]
    .round(2)
)

reconciliation.head()

,transaction_id,order_id,customer_id,amount,payment_date,payment_status,gateway_fee,tax_on_fee,refund_amount,settled_amount,settlement_id,expected_settlement,difference
0,TXN000001,ORD000001,CUST00655,2872.14,2026-08-24,SUCCESS,32.58,5.86,0.00,2833.70,SET000001,2833.70,0.0
1,TXN000002,ORD000002,CUST00282,6197.81,2026-08-05,SUCCESS,124.43,22.40,2526.04,3524.94,SET000002,3524.94,0.0
2,TXN000003,ORD000003,CUST00105,16949.82,2026-08-18,SUCCESS,277.18,49.89,0.00,16622.75,SET000003,16622.75,0.0
3,TXN000004,ORD000004,CUST00090,14803.26,2026-08-02,SUCCESS,225.37,40.57,0.00,14537.32,SET000004,14537.32,0.0
4,TXN000005,ORD000005,CUST00031,2433.01,2026-08-08,SUCCESS,36.04,6.49,0.00,2390.48,SET000005,2390.48,0.0


In [10]:
def classify_transaction(row):

    if pd.isna(row["settled_amount"]):
        return "MISSING_SETTLEMENT"

    if abs(row["difference"]) <= 1:
        return "MATCHED"

    if row["difference"] > 1:
        return "UNDER_SETTLED"

    return "OVER_SETTLED"


reconciliation["status"] = (
    reconciliation.apply(
        classify_transaction,
        axis=1
    )
)

reconciliation[
    [
        "transaction_id",
        "amount",
        "expected_settlement",
        "settled_amount",
        "difference",
        "status"
    ]
].head(20)

,transaction_id,amount,expected_settlement,settled_amount,difference,status
0,TXN000001,2872.14,2833.70,2833.70,0.0,MATCHED
1,TXN000002,6197.81,3524.94,3524.94,0.0,MATCHED
2,TXN000003,16949.82,16622.75,16622.75,0.0,MATCHED
3,TXN000004,14803.26,14537.32,14537.32,0.0,MATCHED
4,TXN000005,2433.01,2390.48,2390.48,0.0,MATCHED
5,TXN000006,15090.27,14753.58,14753.58,0.0,MATCHED
6,TXN000007,17928.89,17709.65,17709.65,0.0,MATCHED
7,TXN000008,10546.04,10364.70,10364.70,0.0,MATCHED
8,TXN000009,7026.95,6821.19,6821.19,0.0,MATCHED
9,TXN000010,17483.67,17086.67,17086.67,0.0,MATCHED


In [11]:
total_transactions = len(reconciliation)

matched = len(
    reconciliation[
        reconciliation["status"] == "MATCHED"
    ]
)

exceptions = (
    total_transactions - matched
)

match_rate = (
    matched / total_transactions
) * 100

unexplained_amount = (
    reconciliation.loc[
        reconciliation["status"] != "MATCHED",
        "difference"
    ]
    .fillna(0)
    .sum()
)

print("===================================")
print("       LEDGERLENS - DAY 1")
print("===================================")

print(
    f"Total Transactions : {total_transactions}"
)

print(
    f"Matched            : {matched}"
)

print(
    f"Exceptions         : {exceptions}"
)

print(
    f"Match Rate         : {match_rate:.2f}%"
)

print(
    f"Unexplained Amount : ₹{unexplained_amount:,.2f}"
)

print("\nException Breakdown:")

print(
    reconciliation["status"]
    .value_counts()
)

       LEDGERLENS - DAY 1
Total Transactions : 2000
Matched            : 1894
Exceptions         : 106
Match Rate         : 94.70%
Unexplained Amount : ₹42,581.13

Exception Breakdown:
status
MATCHED               1894
UNDER_SETTLED           53
MISSING_SETTLEMENT      53
Name: count, dtype: int64


In [12]:
reconciliation.to_csv(
    "reports/reconciliation_report.csv",
    index=False
)

print("Reconciliation report created!")

Reconciliation report created!
